In [2]:
import sys
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import polars as pl
import warnings
from typing import Tuple, Dict, Any

warnings.filterwarnings("ignore")


from mars.feature.binner import MarsOptimalBinner, MarsNativeBinner
from mars.utils.logger import set_log_level, logger

set_log_level("INFO")

def get_memory_usage() -> float:
    """获取当前进程内存占用 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

print(f"✅ 环境就绪 | Polars: {pl.__version__} | Memory: {get_memory_usage():.2f} MB")

✅ 环境就绪 | Polars: 1.34.0 | Memory: 248.73 MB


In [3]:
def run_final_logic_tests():
    print("\n" + "="*60)
    print("🧪 模块 1: 逻辑验证 (索引协议 & Join 路由)")
    print("="*60)

    # --- 1. 数值型：多特殊值与索引协议 ---
    print("\n[Case 1] 数值型：多特殊值索引验证")
    # 构造数据：含两个特殊值 -999, -998
    df_num = pl.DataFrame({"feat": [10, 20, -999, -998, None, 50]})
    y_num = [0, 0, 0, 0, 1, 1]
    
    binner = MarsOptimalBinner(
        features=["feat"], 
        n_bins=2, 
        special_values=[-999, -998],
        missing_values=[None]
    )
    binner.fit(df_num, y_num)
    
    # 验证 Index 模式
    res_idx = binner.transform(df_num, return_type="index")
    indices = res_idx["feat_bin"].to_list()
    print(f"   -> Indices: {indices}")
    # 协议验证：Missing=-1, Special1=-3, Special2=-4
    assert indices[4] == -1, "❌ Missing 索引错误"
    assert indices[2] == -3, "❌ 特殊值1 (-999) 索引错误"
    assert indices[3] == -4, "❌ 特殊值2 (-998) 索引错误"

    # --- 2. 类别型：高基数 Join 路由验证 ---
    print("\n[Case 2] 类别型：高基数 Join 路由测试")
    # 构造超过 join_threshold 的数据
    cats = [f"ID_{i}" for i in range(20)]
    df_cat = pl.DataFrame({"uid": cats})
    y_cat = np.random.randint(0, 2, 20)
    
    # 设置极低的阈值强制触发 Join
    binner_cat = MarsOptimalBinner(
        cat_features=["uid"], 
        n_bins=3, 
        join_threshold=5 
    )
    binner_cat.fit(df_cat, y_cat)
    
    # 验证 Join 后的索引输出
    res_cat = binner_cat.transform(df_cat, return_type="index")
    print(f"   -> Join Index Sample: {res_cat['uid_bin'].head(3).to_list()}")
    assert res_cat["uid_bin"].dtype in [pl.Int16, pl.Int8, pl.Int32], "❌ Join 路由输出类型错误"
    
    # 验证 Label 还原
    res_label = binner_cat.transform(df_cat, return_type="label")
    labels = res_label["uid_bin"].to_list()
    assert isinstance(labels[0], str) and "[" in labels[0], "❌ Join 路由 Label 还原错误"

    # --- 3. 类别型：Other 与 Special 瀑布流 ---
    print("\n[Case 3] 类别型：Other 与 Special 优先级")
    df_mix = pl.DataFrame({"city": ["A", "B", "C", "-999", None]})
    # 训练时只给 A, B
    df_train = pl.DataFrame({"city": ["A"]*10 + ["B"]*10})
    y_train = [1]*10 + [0]*10
    
    binner_mix = MarsOptimalBinner(
        cat_features=["city"], 
        special_values=["-999"]
    )
    binner_mix.fit(df_train, y_train)
    
    res_mix = binner_mix.transform(df_mix, return_type="index")
    mix_indices = res_mix["city_bin"].to_list()
    print(f"   -> Mix Indices: {mix_indices}")
    
    # 验证：A(0), B(1), C(Other:-2), -999(Special:-3), None(Missing:-1)
    assert mix_indices[2] == -2, "❌ 未见类别未归入 Other(-2)"
    assert mix_indices[3] == -3, "❌ 类别型特殊值映射错误"
    assert mix_indices[4] == -1, "❌ 类别型缺失值映射错误"

    # --- 4. 极致防御：类型转换稳定性 ---
    print("\n[Case 4] 类型防御：Int 列含空值")
    df_int = pl.DataFrame({"val": [1, 2, None]}, schema={"val": pl.Int64})
    binner_int = MarsOptimalBinner(features=["val"], n_bins=2)
    binner_int.fit(df_int, [0, 1, 0])
    # 验证不会因为 Int 列调用 is_nan 而崩溃
    try:
        binner_int.transform(df_int)
        print("   ✅ Int Null Safety 通关")
    except Exception as e:
        print(f"   ❌ Int Null Safety 失败: {e}")
        raise e
    
    # --- 5. 极致防御：混合类型配置抗毁性 ---
    print("\n[Case 5] 极致防御：混合类型配置抗毁性")
    print("   场景: 配置了字符串缺失值，但遇到了纯数值列。期望: 自动过滤，不报错。")
    
    df_safe = pl.DataFrame({"age": [20, 30, -999]}, schema={"age": pl.Int64})
    y_safe = [0, 1, 0]
    
    # 故意混入不兼容的类型 "unknown" 和 "N/A"
    binner_safe = MarsOptimalBinner(
        features=["age"], 
        n_bins=2, 
        missing_values=[-999, "unknown", "N/A"],
        special_values=[-1, "Error"]
    )
    
    try:
        binner_safe.fit(df_safe, y_safe)
        res_safe = binner_safe.transform(df_safe, return_type="index")
        
        # 验证 -999 是否正确被识别为 Missing (-1)
        # 如果 _get_safe_values 生效，"unknown" 应该被剔除，-999 保留
        idx_missing = res_safe.filter(pl.col("age") == -999)["age_bin"][0]
        assert idx_missing == -1, f"❌ 混合配置下数值缺失值未识别 (Got {idx_missing})"
        
        print("   ✅ Mixed Type Safety 通关")
    except Exception as e:
        print(f"   ❌ Mixed Type Safety 失败: {e}")
        raise e
    
    print("\n" + "="*60)
    print("🧪 模块 2: 高级边界测试 (统计失真防御 & OptBinning 兼容性)")
    print("="*60)

    # --- 构造一个高缺失率的极端数据集 ---
    # 总行数 10,000
    # 有效行数 1,000 (10%)
    # 期望最小分箱占比 5% (即 500 个样本)
    
    N_TOTAL = 10000
    N_VALID = 1000
    EXPECTED_MIN_SAMPLES = int(0.05 * N_TOTAL) # 500
    
    # 构造数据：前1000个是有效正态分布，后9000个是 None
    rng = np.random.default_rng(42)
    valid_data = rng.normal(0, 1, N_VALID)
    full_data = np.full(N_TOTAL, np.nan, dtype=np.float64)
    full_data[:N_VALID] = valid_data
    
    # 构造 Target：为了让树能切分，构造一个强相关的 y
    # 在有效数据中，大于 0 的是 1，小于 0 的是 0
    y_valid = (valid_data > 0).astype(int)
    y_full = np.concatenate([y_valid, [0] * (N_TOTAL - N_VALID)]) # 填充部分不重要，会被过滤
    
    df_extreme = pl.DataFrame({"feature": full_data}, schema={"feature": pl.Float64})
    
    print(f"\n[Case 6] 决策树预分箱 (NativeBinner) 统计失真防御")
    print(f"   -> 总行数: {N_TOTAL}, 有效行数: {N_VALID}")
    print(f"   -> 设置 min_samples=0.05 (绝对值应为 {EXPECTED_MIN_SAMPLES})")
    
    # 1. 测试 NativeBinner (CART)
    native_binner = MarsNativeBinner(
        features=["feature"],
        method="cart",
        n_bins=5,
        min_samples=0.05, # 这里传入 0.05
    )
    
    native_binner.fit(df_extreme, y_full)
    
    cuts = native_binner.bin_cuts_["feature"]
    # 预期逻辑：
    # 有效数据 1000 行。
    # 绝对阈值 = 10000 * 0.05 = 500 行。
    # 理论上 1000 行数据最多只能切成 [500, 500] 两个叶子节点（甚至可能无法切分，取决于具体的树生长）。
    # 如果逻辑错误（基于 1000 行算 5% = 50行），树会切得很碎（5个箱子）。
    # 如果逻辑正确（基于 10000 行算 5% = 500行），树最多切 1 刀，或者不切。
    
    print(f"   -> NativeBinner 生成切点数: {len(cuts) - 2} (不含 inf)")
    print(f"   -> 切点详情: {cuts}")
    
    # 验证：切点数应该很少（0 或 1），绝对不能是 4 (即5箱)
    assert len(cuts) <= 3, f"❌ NativeBinner 过拟合！在仅有 1000 有效样本且要求 min=500 的情况下切出了 {len(cuts)-2} 个点。"
    print("   ✅ NativeBinner 统计失真防御测试通过")


    print(f"\n[Case 7] 最优分箱 (OptimalBinning) 接口兼容性 & 水位熔断")
    # 2. 测试 OptimalBinning
    # 我们使用同样的极端数据，但这次要求更严苛
    # 尝试触发 "水位熔断"：要求 min_bin_size=0.06 (600人)
    # 此时有效数据 1000 人。如果切分，最小两箱加起来需要 1200 人 > 1000。
    # 预期：应该触发水位熔断，直接返回预分箱结果（或者单箱）。
    
    opt_binner = MarsOptimalBinner(
        features=["feature"],
        n_bins=5,
        min_n_bins=2,
        min_bin_size=0.06, # 6% * 10000 = 600 人
        prebinning_method="quantile", # 预分箱用简单的
        n_prebins=10
    )
    
    opt_binner.fit(df_extreme, y_full)
    opt_cuts = opt_binner.bin_cuts_["feature"]
    
    print(f"   -> 设置 min_bin_size=0.06 (600人)。有效数据 1000 人。")
    print(f"   -> 理论需求: 2箱 * 600 = 1200 > 1000。应触发熔断。")
    print(f"   -> OptBinner 最终切点: {opt_cuts}")
    
    # 如果触发了熔断，它会回退到预分箱结果。
    # 或者如果进入了 Worker 内部的 check，会直接 return fallback。
    # 关键是不能报错（ValueError: min_bin_size must be in (0, 0.5]）。
    
    if len(opt_cuts) > 2:
        print("   ⚠️ 警告: 似乎进行了切分（可能是预分箱的结果）。检查日志确认是否触发了 OptBinning 求解。")
    
    # 3. 验证整数转换逻辑 (正常场景)
    # 构造一个适中的场景，确保 OptBinning 会被调用，且传入的 ratio 合法
    # 总行数 1000，有效 1000，min_bin_size=0.05 (50)
    # 预期：OptBinning 正常运行，不报 ValueError
    print(f"\n[Case 8] 最优分箱：正常整数转换逻辑验证")
    df_normal = pl.DataFrame({"x": rng.normal(0, 1, 1000)})
    y_normal = (df_normal["x"] > 0).to_numpy().astype(int)
    
    opt_binner_norm = MarsOptimalBinner(
        features=["x"],
        min_bin_size=0.05, # 传入 float，内部转 int 再转 ratio
        n_jobs=1
    )
    try:
        opt_binner_norm.fit(df_normal, y_normal)
        print("   ✅ OptBinner 正常场景运行无报错")
        print(f"   -> 最终切点数: {len(opt_binner_norm.bin_cuts_['x']) - 2}")
    except Exception as e:
        print(f"   ❌ OptBinner 失败: {e}")
        # 重点检查是否报了 min_bin_size 必须在 (0, 0.5] 的错
        if "min_bin_size must be in (0, 0.5]" in str(e):
            print("   ❌ 关键错误：未正确将绝对值转换为切片比例！")
        raise e

    print("\n🎉 高级边界测试全部通过！统计防御与接口兼容性验证完成。")
    
    print("\n" + "="*60)
    print("🧪 模块 3: 约束冲突测试 (min_n_bins > pre_bins)")
    print("="*60)

    print(f"\n[Case 9] 最小箱数约束冲突")
    print("   场景: 数据只有两个唯一值 [0, 1]，物理上只能分 2 箱。")
    print("   约束: 设置 min_n_bins=3。")
    print("   预期: 求解器无法满足约束 -> 状态非 OPTIMAL -> 自动回退到预分箱结果 (2箱)。")

    # 1. 构造极简二值数据 (只有 0 和 1)
    # 这种数据无论怎么切，预分箱最多只能切出 [-inf, 0.5, inf] 两个箱子
    df_binary = pl.DataFrame({"feature": [0]*500 + [1]*500})
    y_binary = [0]*500 + [1]*500 # 强相关 y

    # 2. 初始化分箱器，强行要求分 3 箱
    binner = MarsOptimalBinner(
        features=["feature"],
        n_bins=5,
        min_n_bins=3, # <--- 冲突点：数据极限是2，要求3
        prebinning_method="quantile",
    )

    # 3. 执行拟合
    try:
        binner.fit(df_binary, y_binary)
        cuts = binner.bin_cuts_["feature"]
        
        print(f"   -> 最终切点: {cuts}")
        print(f"   -> 最终箱数: {len(cuts) - 1}")
        
        # 4. 验证
        # 此时应该只有 2 个箱子 (3个切点: -inf, split, inf)
        if len(cuts) == 3:
            print("   ✅ 成功触发 Fallback，返回了物理极限的 2 箱。")
        elif len(cuts) < 3:
            print("   ⚠️ 警告: 甚至连 2 箱都没分出来 (可能是单调性或其他约束导致)。")
        else:
            print(f"   ❌ 错误: 竟然分出了 {len(cuts)-1} 箱？这对于二值数据是不可能的。")
            
        # 检查是否记录了失败原因 (可选)
        if "feature" in binner.fit_failures_:
            print(f"   ℹ️ 内部日志记录: {binner.fit_failures_['feature']}")
            
    except Exception as e:
        print(f"   ❌ 程序崩溃: {e}")
        raise e

    print("\n🎉 最终版逻辑验证全部通过！代码已达极致工业级水准。")

run_final_logic_tests()


🧪 模块 1: 逻辑验证 (索引协议 & Join 路由)

[Case 1] 数值型：多特殊值索引验证
[MARS] 2026-01-23 00:05:43 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 0.0223s
[MARS] 2026-01-23 00:05:45 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 1.8743s
[MARS] 2026-01-23 00:05:45 - WARNING - ⚠️ MarsOptimalBinner: 1 features encountered issues (1 num, 0 cat). Fallback applied. Check `.fit_failures_` for details. Sample: [('feat', 'Low variance or insufficient samples')]
   -> Indices: [0, 0, -3, -4, -1, 1]

[Case 2] 类别型：高基数 Join 路由测试
   -> Join Index Sample: [0, 0, 0]

[Case 3] 类别型：Other 与 Special 优先级
   -> Mix Indices: [0, 0, -2, -3, -1]

[Case 4] 类型防御：Int 列含空值
[MARS] 2026-01-23 00:05:48 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 0.0247s
[MARS] 2026-01-23 00:05:49 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 1.2582s
[MARS] 2026-01-23 00:05:49 - WARNING - ⚠️ MarsOptimalBinner: 1 features encountered issues (1 num, 0 cat). Fallback applied. Check `.fit_failures_` for deta

In [ ]:
import numpy as np
import polars as pl
from datetime import datetime

# ==========================================
# 1. 模拟全类型数据集 (5000行样本)
# ==========================================
n_rows = 5000
np.random.seed(42)

# 特征 A: 强正相关数值 (随着值增大，风险增高)
feat_pos = np.random.normal(0, 1, n_rows)
# 特征 B: 强负相关数值 (随着值增大，风险降低，测试 AUC 修正逻辑)
feat_neg = np.random.normal(0, 1, n_rows)
# 特征 C: 随机噪声 (IV/KS 应该接近 0)
feat_noise = np.random.normal(0, 1, n_rows)
# 特征 D: 含特殊值 (-999) 和 缺失值 (null) 的数值特征
feat_special = np.random.choice([10.0, 20.0, 30.0, -999.0, np.nan], n_rows, p=[0.3, 0.3, 0.2, 0.1, 0.1])
# 特征 E: 类别型特征 (A类高风险，B类低风险)
feat_cat = np.random.choice(['High_Risk', 'Low_Risk', 'Medium_Risk'], n_rows, p=[0.2, 0.5, 0.3])

# 构造目标变量 y (基于逻辑回归公式)
# 线性组合：pos越大y越趋向1，neg越大y越趋向0，cat='High_Risk'时y趋向1
logits = 1.5 * feat_pos - 2.0 * feat_neg + (np.where(feat_cat == 'High_Risk', 1.0, -1.0))
prob = 1 / (1 + np.exp(-logits))
y = (prob > np.random.rand(n_rows)).astype(int)

# 封装为 Polars DataFrame
df = pl.DataFrame({
    "num_pos": feat_pos,
    "num_neg": feat_neg,
    "num_noise": feat_noise,
    "num_special_missing": feat_special,
    "cat_risk": feat_cat,
    "target": y
})

print(f"✅ 测试数据构建完成，形状: {df.shape}")

# ==========================================
# 2. 初始化分箱器并拟合 (MarsOptimalBinner)
# ==========================================
# 我们定义 -999 为特殊值，None 会自动识别为缺失值
binner = MarsOptimalBinner(
    n_bins=5,
    special_values=[-999],
    missing_values=[np.nan],
    cat_features=["cat_risk"]
)

# 执行拟合
binner.fit(df, y=df["target"])

# ==========================================
# 3. 核心：产出分箱性能画像 (Indicator Profiling)
# ==========================================
# 我们开启 update_woe=True，这样计算后的权重会存入 binner 内部供转换使用
report = binner.profile_bin_performance(df, y=df["target"], update_woe=True)

# ==========================================
# 4. 结果查看与解读
# ==========================================
# 设置 Polars 显示选项，方便在 Jupyter 查看宽表
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(20)

print("\n" + "="*50)
print("📊 分箱指标计算全量报告 (按特征分组汇总)")
print("="*50)

# 我们对报告进行二次加工，方便一眼看出哪些特征强，哪些特征弱
summary = (
    report.group_by("feature")
    .agg([
        pl.col("IV").first(),
        pl.col("KS").first(),
        pl.col("AUC").first()
    ])
    .sort("IV", descending=True)
)
print(summary)

print("\n" + "="*50)
print("🔍 详细分箱明细 (展示正相关、负相关、特殊值处理效果)")
print("="*50)

# 查看特征 num_pos (正相关) 和 num_special_missing (含特殊值) 的明细
target_show = ["num_pos_bin", "num_neg_bin", "num_special_missing_bin", "cat_risk_bin"]
display_detail = report.filter(pl.col("feature").is_in(target_show))
display(display_detail)

# ==========================================
# 5. 验证持久化逻辑 (Persistence Test)
# ==========================================
print("\n" + "="*50)
print("💾 持久化测试: to_dict -> from_dict -> transform")
print("="*50)

# 导出字典
binner_dict = binner.to_dict()

# 从字典恢复新实例 (模拟推理环境)
new_binner = MarsOptimalBinner.from_dict(binner_dict)

# 在新实例上进行转换测试
test_X = df.head(5)
# 转换出 WOE 值
woe_res = new_binner.transform(test_X, return_type="woe")

print("▶️ 推理实例转换结果 (WOE 后缀列):")
display(woe_res.select(pl.col("^.*_woe$")))

# 检查转换出来的 WOE 是否在之前 report 的明细里能对上
sample_feat = "num_pos"
sample_woe = woe_res[0, f"{sample_feat}_woe"]
print(f"✅ 随机验证: 特征 {sample_feat} 的首行 WOE 值为 {sample_woe:.4f}")

In [ ]:
(
    report
    .group_by("feature")
    .sum()
)

In [ ]:
import polars as pl

def test_categorical_trap():
    print("🔥 Polars 'Categorical Trap' 现场还原实验 🔥\n")

    # 1. 构造简单数据
    df = pl.DataFrame({"age": [18, 55, 18, 55]})
    
    # 2. 执行分箱 (Cut)
    # 关键点：我们将标签设为 "100" 和 "200"，而不是默认的 "0", "1"
    # 这样如果结果出现 0 或 1，你就知道它在“读编号”而不是“读内容”了
    breaks = [30.0]
    labels = ["100", "200"]  # 假设这是我们的业务映射值
    
    df_cut = df.with_columns(
        pl.col("age").cut(breaks, labels=labels, left_closed=True).alias("cat_col")
    )
    
    print("--- 1. 原始分箱结果 (Categorical 类型) ---")
    print(df_cut)
    
    # 3. ❌ 错误做法：直接转 Int
    # 这相当于偷看了 Polars 的“内部账本”
    df_wrong = df_cut.select(
        pl.col("cat_col"),
        pl.col("cat_col").cast(pl.Int16).alias("physical_index_wrong")
    )
    
    print("\n--- 2. ❌ 错误做法 (.cast(pl.Int16)) ---")
    print("现象：你看到的是 0 和 1，而不是我们定义的 100 和 200。")
    print("原因：这是 Polars 内部分配的存储 ID (Physical Index)。")
    print("后果：如果你的 mapping 字典是 {100: 'Young', 200: 'Old'}，用 0 和 1 去查肯定报错！")
    print(df_wrong)

    # 4. ✅ 正确做法：先转 String 再转 Int
    # 这相当于强制 Polars 把“内容”吐出来
    df_right = df_cut.select(
        pl.col("cat_col"),
        pl.col("cat_col").cast(pl.Utf8).cast(pl.Int16).alias("logical_value_right")
    )
    
    print("\n--- 3. ✅ 正确做法 (.cast(pl.Utf8).cast(pl.Int16)) ---")
    print("现象：成功拿到了 100 和 200。")
    print("原因：先转字符串提取了'标签文本'，再转数字得到了'业务数值'。")
    print(df_right)

    # 5. 💀 模拟“索引偏移” (为什么你会看到 3 ?)
    # Polars 的物理索引是从 0 开始排的。
    # 如果你的数据里 labels=["0", "1", "2"]
    # 正常情况下： "0"->0, "1"->1, "2"->2 (看起来没问题)
    # 
    # 但如果 Polars 内部决定保留 0 给 Null 值 (或某种特殊对其)，它可能会变成：
    # Null -> 0
    # "0"  -> 1
    # "1"  -> 2
    # "2"  -> 3  <-- 💥 你的 mapping 里只有 2，但它给了你 3！
    
    print("\n--- 4. 💀 为什么之前你的代码会出现 '3' ? ---")
    print("当数据变复杂时，Polars 可能会把物理索引向后推移 (Off-by-one)。")
    print("比如标签 '2' 的物理索引变成了 3。")
    print("如果你用旧代码直接取 ID (拿到3)，去查只有 {0,1,2} 的字典，自然就匹配失败，原来的 3 就直接显示出来了。")
    print("而用新代码，无论它内部排到几号，我们读出来的永远是字符串 '2'。")

if __name__ == "__main__":
    test_categorical_trap()

In [ ]:
# ==========================================
# 模块 3: 大规模压力测试 (20万行 x 1000列 - 三模式对比)
# ==========================================

N_ROWS = 200_000
N_COLS = 1000

def run_stress_test():
    print("\n" + "="*80)
    print(f"🔥 模块 3: 大规模压力测试 ({N_ROWS:,}行 x {N_COLS}列)")
    print("="*80)
    
    # 1. 生成海量数据 (Float32 节省内存)
    print(f"🚀 [DataGen] 正在生成数据...")
    data_matrix = np.random.randn(N_ROWS, N_COLS).astype(np.float32)
    y = (data_matrix[:, 0] > 0).astype(int)
    df = pl.from_numpy(data_matrix, schema=[f"f_{i}" for i in range(N_COLS)])
    print(f"✅ 数据准备完毕 | 内存占用: {get_memory_usage():.2f} MB")

    results = []
    methods = [
        # "quantile", 
        # "uniform",
        "cart"
        ]

    for method in methods:
        print(f"\n🧪 Testing Pre-binning Method: [{method.upper()}]")
        print("-" * 45)
        
        # 清理内存
        gc.collect()
        time.sleep(1) # 给系统回收缓冲的时间
        mem_start = get_memory_usage()
        t_start = time.time()
        
        # 2. 训练 MarsOptimalBinner
        # 混合动力引擎：Stage 1 (method) -> Stage 2 (CP Solver)
        binner = MarsOptimalBinner(
            n_bins=5, 
            n_prebins=50, 
            prebinning_method=method,
            n_jobs=-1,      
            time_limit=2    
        )
        
        binner.fit(df, y)
        t_fit = time.time() - t_start
        
        # 3. 转换 (默认返回 index 模式以测试极致性能)
        t_trans_start = time.time()
        res = binner.transform(df, return_type="index")
        # 强制触发 Polars 计算
        _ = res.select(pl.col("f_0_bin")).height 
        t_trans = time.time() - t_trans_start
        
        mem_peak = get_memory_usage() - mem_start
        
        results.append({
            "Method": method,
            "Fit Time(s)": t_fit,
            "Trans Time(s)": t_trans,
            "Mem Gain(MB)": mem_peak,
            "Throughput(f/s)": N_COLS / t_fit
        })
        
        print(f"   ⏱️  Fit Time:       {t_fit:.2f} s")
        print(f"   🚀  Trans Time:     {t_trans:.2f} s")
        print(f"   💾  Mem Overhead:   {mem_peak:.2f} MB")

    # 4. 打印汇总看板
    print("\n" + "🏆 STRESS TEST SUMMARY".center(60, "="))
    report_df = pd.DataFrame(results)
    display(report_df.round(3))
    
    # 彻底清理
    del df, data_matrix; gc.collect()

run_stress_test()

In [2]:
import sys
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import polars as pl
import warnings
from typing import Tuple, Dict, Any

warnings.filterwarnings("ignore")


from mars.feature.binner import MarsOptimalBinner, MarsNativeBinner
from mars.utils.logger import set_log_level, logger

set_log_level("INFO")

def get_memory_usage() -> float:
    """获取当前进程内存占用 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

print(f"✅ 环境就绪 | Polars: {pl.__version__} | Memory: {get_memory_usage():.2f} MB")
from optbinning import BinningProcess

def benchmark_optimal_binning_with_check():
    # ==========================================
    # 1. 数据生成 (模拟大宽表场景)
    # ==========================================
    N_ROWS = 200_000
    N_COLS = 500
    MISSING_RATE = 0.5  # 10% 的缺失率
    
    print(f"🛠️ 正在生成测试数据: {N_ROWS} 行 x {N_COLS} 列...")
    np.random.seed(42)
    X = np.random.randn(N_ROWS, N_COLS).astype(np.float32)
    
    # 构造 Target: 简单的线性关系 + 噪声
    # 这种构造方式保证了特征具有一定的单调性趋势，利于 OptimalBinning 求解
    y = (X[:, 0] * 5 + X[:, 1] * 2 + np.random.randn(N_ROWS) > 0).astype(int)
    
    # 【新增】引入随机缺失值
    # 创建一个随机掩码，将 10% 的位置置为 NaN
    mask = np.random.random((N_ROWS, N_COLS)) < MISSING_RATE
    X[mask] = np.nan
    
    feat_names = [f"f_{i}" for i in range(N_COLS)]
    
    # 准备两种格式的数据
    df_pl = pl.from_numpy(X, schema=feat_names) # 给 Mars 用
    df_pd = pd.DataFrame(X, columns=feat_names) # 给 OptBinning 用
    
    print("✅ 数据准备完毕。\n")

    # ==========================================
    # 2. 定义测试配置
    # ==========================================
    methods = [
        # "quantile", 
        # "uniform", 
        "cart"
        ]
    
    print(f"{'Method':<10} | {'Engine':<12} | {'Fit (s)':<10} | {'Trans (s)':<10} | {'Total (s)':<10} | {'Speedup':<8}")
    print("-" * 80)

    for method in methods:
        # ==========================================
        # Round 1: Mars (Hybrid Engine)
        # ==========================================
        gc.collect()
        t0 = time.time()
        
        mars_model = MarsOptimalBinner(
            n_bins=5, 
            n_prebins=50,       
            prebinning_method=method, 
            min_prebin_size=0.05,
            n_jobs=-1,
            time_limit=2        
        )
        
        mars_model.fit(df_pd, y)
        print(mars_model.fit_failures_)
        t_fit_end = time.time()
        
        res = mars_model.transform(df_pd)
        # _ = res.select(pl.col(f"f_0_bin")).to_series() 
        print(type(res))
        t_trans_end = time.time()
        
        mars_fit_time = t_fit_end - t0
        mars_trans_time = t_trans_end - t_fit_end
        mars_total = mars_fit_time + mars_trans_time

        print(f"{method:<10} | {'Mars':<12} | {mars_fit_time:<10.4f} | {mars_trans_time:<10.4f} | {mars_total:<10.4f} | {'1.0x':<8}")

        # ==========================================
        # Round 2: OptBinning (Standard)
        # ==========================================
        gc.collect()
        t0 = time.time()
        
        # 1. 构造详细的参数配置字典
        fit_params = {}
        for col in feat_names:
            fit_params[col] = {
                # --- 预分箱参数 ---
                'prebinning_method': method, 
                'max_n_prebins': 50,
                
                # --- [关键] 核心优化参数 (对齐 Mars) ---
                # 强制 OptBinning 也进行双向搜索 (升序+降序)，而不是默认的猜方向
                'monotonic_trend': 'auto_asc_desc',  
                
                # 最小箱占比 (虽然后面 BinningProcess 也有这个参数，但写在这里优先级最高)
                'min_bin_size': 0.05,
                "min_prebin_size": 0.05,
            }

        # 2. 初始化 BinningProcess
        opt_process = BinningProcess(
            variable_names=feat_names,
            max_n_bins=5,  
            n_jobs=-1,  
            binning_fit_params=fit_params 
        )
        
        opt_process.fit(df_pd, y)
        t_fit_end = time.time()
        
        _ = opt_process.transform(df_pd, metric="indices")
        t_trans_end = time.time()
        
        opt_fit_time = t_fit_end - t0
        opt_trans_time = t_trans_end - t_fit_end
        opt_total = opt_fit_time + opt_trans_time
        
        speedup = opt_total / mars_total
        print(f"{method:<10} | {'OptBinning':<12} | {opt_fit_time:<10.4f} | {opt_trans_time:<10.4f} | {opt_total:<10.4f} | {speedup:<8.1f}x")

        # ==========================================
        # 🔍 切点一致性抽检 (Check Splits) 冒烟测试 
        # ==========================================
        print(f"   [🔍 切点抽样对比: {method}]")
        
        # 随机抽取 3 个特征
        check_cols = np.random.choice(feat_names, 3, replace=False)
        
        for col in check_cols:
            # 1. 获取 Mars 切点 (去除 -inf, inf)
            if col in mars_model.bin_cuts_:
                mars_cuts = mars_model.bin_cuts_[col]
                # Mars 的格式是 [-inf, c1, c2, ..., inf]，比较时去掉无穷大
                mars_splits = [c for c in mars_cuts if c != float('-inf') and c != float('inf')]
            else:
                mars_splits = []

            # 2. 获取 OptBinning 切点
            # OptBinning 对象可能没有生成 splits (比如单调性约束无法满足)，需要处理异常
            try:
                opt_bin_obj = opt_process.get_binned_variable(col)
                if opt_bin_obj is not None:
                    opt_splits = opt_bin_obj.splits.tolist()
                else:
                    opt_splits = []
            except Exception:
                opt_splits = []
            
            # 3. 打印对比
            # 保留4位小数以便观察
            m_str = ", ".join([f"{x:.4f}" for x in mars_splits])
            o_str = ", ".join([f"{x:.4f}" for x in opt_splits])
            
            # 简单判断是否完全一致 (允许极小误差)
            is_close = False
            if len(mars_splits) == len(opt_splits):
                if len(mars_splits) == 0:
                    is_close = True
                else:
                    is_close = np.allclose(mars_splits, opt_splits, atol=1e-4)
            
            status_icon = "✅" if is_close else "⚠️"
            
            print(f"   - Feature {col:<5}: {status_icon}")
            print(f"     Mars: [{m_str}]")
            print(f"     Opt : [{o_str}]")
        
        print("-" * 80)

if __name__ == "__main__":
    benchmark_optimal_binning_with_check()

✅ 环境就绪 | Polars: 1.34.0 | Memory: 1280.38 MB
🛠️ 正在生成测试数据: 200000 行 x 500 列...
✅ 数据准备完毕。

Method     | Engine       | Fit (s)    | Trans (s)  | Total (s)  | Speedup 
--------------------------------------------------------------------------------
[MARS] 2026-01-25 19:45:47 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 2.5294s
[MARS] 2026-01-25 19:45:49 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 4.7416s
[MARS] 2026-01-25 19:45:49 - WARNING - ⚠️ MarsOptimalBinner: 2 features encountered issues (2 num, 0 cat). Fallback applied. Check `.fit_failures_` for details. Sample: [('f_129', 'Solver collapsed to single bin; fallback to pre-bins'), ('f_287', 'Solver collapsed to single bin; fallback to pre-bins')]
{'f_129': 'Solver collapsed to single bin; fallback to pre-bins', 'f_287': 'Solver collapsed to single bin; fallback to pre-bins'}
<class 'pandas.core.frame.DataFrame'>
cart       | Mars         | 4.8961     | 0.6095     | 5.5055     | 1.0x    
cart       | OptBi